# Hourly Location-Level Rental Aggregation and Feature Engineering

The goal of this notebook is to transform raw bike rental event records into a location-level hourly demand dataset. Each row represents the rental demand for one location during one hourly time window.

The dataset is enriched with time-based features, hourly weather information, and holiday information.

The final output includes two CSV files:

1. A full version with the holiday name column for checking and validation.
2. A model-ready version that only keeps the numeric `is_holiday` feature.

The notebook is structured as follow:

1. Prepare rental datasets
2. Merge registered and direct rental data
3. Merge with weather data
4. Merge with holiday data
5. Save final dataset

## 1. Prepare rental datasets
### 1.1 Load and convert datetime

In [75]:
# this has been done in the first part
import pandas as pd

registered = pd.read_csv("../data/registered_bike_rentals.csv")
direct = pd.read_csv("../data/direct_pickup_bike_rentals.csv")
registered["datetime"] = pd.to_datetime(registered["datetime"])
direct["datetime"] = pd.to_datetime(direct["datetime"])

### 1.2 Round datetime to hour

In [76]:
# create "hour"feature by flooring timestamp to the beginning of its hour
registered["hour"] = registered["datetime"].dt.floor("h")
direct["hour"] = direct["datetime"].dt.floor("h")

## 2. Merge registered and direct data
### 2.1 Combine all rental events

In [77]:
registered_events = registered[["hour", "location_id"]].copy()
registered_events["rental_type"] = "registered"

direct_events = direct[["hour", "location_id"]].copy()
direct_events["rental_type"] = "direct"

rental_events = pd.concat(
    [registered_events, direct_events],
    ignore_index=True
)
rental_events.head()

,hour,location_id,rental_type
0,2011-01-01,16,registered
1,2011-01-01,18,registered
2,2011-01-01,18,registered
3,2011-01-01,9,registered
4,2011-01-01,11,registered


### 2.2 group up by hour and location

In [78]:
hourly_location_rentals = (
    rental_events
    .groupby(["hour", "location_id", "rental_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
hourly_location_rentals.head()
# show missing count

rental_type,hour,location_id,direct,registered
0,2011-01-01,2,1,0
1,2011-01-01,3,0,1
2,2011-01-01,4,0,2
3,2011-01-01,5,1,0
4,2011-01-01,9,0,1


In [79]:
# fill up missing columns with 0
if "registered" not in hourly_location_rentals.columns:
    hourly_location_rentals["registered"] = 0

if "direct" not in hourly_location_rentals.columns:
    hourly_location_rentals["direct"] = 0

hourly_location_rentals = hourly_location_rentals.rename(columns={"registered": "registered_count", "direct": "direct_count"})

# create the total rental count
hourly_location_rentals["total_count"] = (hourly_location_rentals["registered_count"] + hourly_location_rentals["direct_count"])

hourly_location_rentals = (hourly_location_rentals.sort_values(["hour", "location_id"]).reset_index(drop=True))
hourly_location_rentals.head()

rental_type,hour,location_id,direct_count,registered_count,total_count
0,2011-01-01,2,1,0,1
1,2011-01-01,3,0,1,1
2,2011-01-01,4,0,2,2
3,2011-01-01,5,1,0,1
4,2011-01-01,9,0,1,1


### 2.2 Handle missing hourly records
From the exploration part we have already known that, there are missing hourly records in the rental data. Therefore we create a full time table, find the specific rows and fill the values with 0.

In [80]:
print("Index number before handling:", hourly_location_rentals.shape[0])

Index number before handling: 312386


In [ ]:
# create full hour-location table
full_hours = pd.DataFrame({
    "hour": pd.date_range(
        start=hourly_location_rentals["hour"].min(),
        end=hourly_location_rentals["hour"].max(),
        freq="h"
    )
})

all_locations = pd.DataFrame({
    "location_id": rental_events["location_id"].drop_duplicates().sort_values()
})

full_hour_location = full_hours.merge(
    all_locations,
    how="cross"
)

#merge
hourly_location_rentals = full_hour_location.merge(
    hourly_location_rentals,
    on=["hour", "location_id"],
    how="left"
)

# mark missing rows
hourly_location_rentals["is_missing_rental"] = (hourly_location_rentals["total_count"].isna().astype(int))


# fill up
count_cols = ["registered_count", "direct_count", "total_count"]
hourly_location_rentals[count_cols] = (
    hourly_location_rentals[count_cols]
    .fillna(0)
    .astype(int)
)

In [82]:
print("Expected rows from full_hours:", full_hour_location.shape[0])
print("Index number after handling:", hourly_location_rentals.shape[0])

Expected rows from full_hours: 368424
Index number after handling: 368424


### 2.3 Derive time-based features

In [83]:
hourly_location_rentals["date"] = hourly_location_rentals["hour"].dt.date  # for holidays merging
hourly_location_rentals["hour_of_day"] = hourly_location_rentals["hour"].dt.hour # demand related: rush hour, lunch time..
hourly_location_rentals["day_of_week"] = hourly_location_rentals["hour"].dt.dayofweek # weekdays and weekends
hourly_location_rentals["month"] = hourly_location_rentals["hour"].dt.month # seasonal demand
hourly_location_rentals["is_weekend"] = hourly_location_rentals["day_of_week"].isin([5, 6]).astype(int)

hourly_location_rentals.head()

,hour,location_id,direct_count,registered_count,total_count,missing_rental,date,hour_of_day,day_of_week,month,is_weekend
0,2011-01-01,0,0,0,0,1,2011-01-01,0,5,1,1
1,2011-01-01,1,0,0,0,1,2011-01-01,0,5,1,1
2,2011-01-01,2,1,0,1,0,2011-01-01,0,5,1,1
3,2011-01-01,3,0,1,1,0,2011-01-01,0,5,1,1
4,2011-01-01,4,0,2,2,0,2011-01-01,0,5,1,1


In [84]:
hourly_location_rentals.info()

<class 'pandas.DataFrame'>
RangeIndex: 368424 entries, 0 to 368423
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   hour              368424 non-null  datetime64[us]
 1   location_id       368424 non-null  int64         
 2   direct_count      368424 non-null  int64         
 3   registered_count  368424 non-null  int64         
 4   total_count       368424 non-null  int64         
 5   missing_rental    368424 non-null  int64         
 6   date              368424 non-null  object        
 7   hour_of_day       368424 non-null  int32         
 8   day_of_week       368424 non-null  int32         
 9   month             368424 non-null  int32         
 10  is_weekend        368424 non-null  int64         
dtypes: datetime64[us](1), int32(3), int64(6), object(1)
memory usage: 26.7+ MB


In [85]:
hourly_location_rentals.isna().sum() # no missing value

hour                0
location_id         0
direct_count        0
registered_count    0
total_count         0
missing_rental      0
date                0
hour_of_day         0
day_of_week         0
month               0
is_weekend          0
dtype: int64

In [86]:
print("Original registered records:", len(registered))
print("Aggregated registered count:", hourly_location_rentals["registered_count"].sum())

print("\nOriginal direct records:", len(direct))
print("Aggregated direct count:", hourly_location_rentals["direct_count"].sum())

print("\nOriginal total records:", len(registered) + len(direct))
print("Aggregated total count:", hourly_location_rentals["total_count"].sum())

Original registered records: 2672662
Aggregated registered count: 2672662

Original direct records: 620017
Aggregated direct count: 620017

Original total records: 3292679
Aggregated total count: 3292679


## 3. Merge with weather data
### 3.1 Select weather features and create a clean table

In [87]:
weather = pd.read_csv("../data/weather.csv")
weather["datetime"] = pd.to_datetime(weather["datetime"])

# create same column "hour"
weather["hour"] = weather["datetime"].dt.floor("h")

# remove unnecessary features
weather_features = weather.drop(columns=["id", "datetime"])

# create full hours weather table
weather_features = full_hours.merge(weather_features, on="hour", how="left")

# weather_features = weather_features.sort_values("hour").reset_index(drop=True)
weather_features.head()

,hour,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,2011-01-01 00:00:00,clear,3.3,3.0,81.0,0.0
1,2011-01-01 01:00:00,clear,2.3,2.0,80.0,0.0
2,2011-01-01 02:00:00,clear,2.3,2.0,80.0,0.0
3,2011-01-01 03:00:00,clear,3.3,3.0,75.0,0.0
4,2011-01-01 04:00:00,clear,3.3,3.0,75.0,0.0


### 3.2 Handle missing weather values

In [ ]:
weather_columns = [
    col for col in weather_features.columns
    if col != "hour"
]

weather_features["is_missing_weather"] = (weather_features[weather_columns].isna().any(axis=1).astype(int))

print("Rows with missing weather value:", weather_features["is_missing_weather"].sum())
print(weather_features[weather_columns].isna().sum())

Rows with missing weather value: 165
conditions                 165
temperature_c              165
perceived_temperature_c    165
humidity                   165
windspeed_kmh              165
dtype: int64


In [89]:
# handle numerical features with linear interpolation
numeric_weather_cols = (
    weather_features[weather_columns]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

weather_features[numeric_weather_cols] = (
    weather_features[numeric_weather_cols]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)

In [90]:
# handle categorical features with value forward or backward
categorical_weather_cols = (
    weather_features[weather_columns]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

weather_features[categorical_weather_cols] = (
    weather_features[categorical_weather_cols]
    .ffill()
    .bfill()
)
print(weather_features[weather_columns].isna().sum())

conditions                 0
temperature_c              0
perceived_temperature_c    0
humidity                   0
windspeed_kmh              0
dtype: int64


### 3.3 Merge with hourly_location_rentals

In [91]:
rentals_with_weather = hourly_location_rentals.merge(
    weather_features,
    on="hour",
    how="left"
)
rentals_with_weather.head()

,hour,location_id,direct_count,registered_count,total_count,missing_rental,date,hour_of_day,day_of_week,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,missing_weather
0,2011-01-01,0,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0
1,2011-01-01,1,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0
2,2011-01-01,2,1,0,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0
3,2011-01-01,3,0,1,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0
4,2011-01-01,4,0,2,2,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0


In [92]:
rentals_with_weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 368424 entries, 0 to 368423
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   hour                     368424 non-null  datetime64[us]
 1   location_id              368424 non-null  int64         
 2   direct_count             368424 non-null  int64         
 3   registered_count         368424 non-null  int64         
 4   total_count              368424 non-null  int64         
 5   missing_rental           368424 non-null  int64         
 6   date                     368424 non-null  object        
 7   hour_of_day              368424 non-null  int32         
 8   day_of_week              368424 non-null  int32         
 9   month                    368424 non-null  int32         
 10  is_weekend               368424 non-null  int64         
 11  conditions               368424 non-null  str           
 12  temperature_c            36

Missing weather values are handled differently from missing rental counts. Numeric weather columns such as `temperature_c`, `perceived_temperature_c`, `humidity`, `windspeed_kmh` are filled using time-based interpolation, while categorical weather columns like `conditions` are filled using forward and backward filling.

## 4. Merge with holiday data

In [93]:
holidays = pd.read_csv("../data/holidays.csv")
holidays["date"] = pd.to_datetime(holidays["date"]).dt.date #convert to same datatype "object" as main data

final_data = rentals_with_weather.merge(
    holidays[["date", "holiday"]],
    on="date",
    how="left"
)

final_data["is_holiday"] = final_data["holiday"].notna().astype(int)

## 5. Check final data

In [94]:
final_data.head()

,hour,location_id,direct_count,registered_count,total_count,missing_rental,date,hour_of_day,day_of_week,month,is_weekend,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,missing_weather,holiday,is_holiday
0,2011-01-01,0,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0
1,2011-01-01,1,0,0,0,1,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0
2,2011-01-01,2,1,0,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0
3,2011-01-01,3,0,1,1,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0
4,2011-01-01,4,0,2,2,0,2011-01-01,0,5,1,1,clear,3.3,3.0,81.0,0.0,0,NaN,0


In [95]:
final_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 368424 entries, 0 to 368423
Data columns (total 19 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   hour                     368424 non-null  datetime64[us]
 1   location_id              368424 non-null  int64         
 2   direct_count             368424 non-null  int64         
 3   registered_count         368424 non-null  int64         
 4   total_count              368424 non-null  int64         
 5   missing_rental           368424 non-null  int64         
 6   date                     368424 non-null  object        
 7   hour_of_day              368424 non-null  int32         
 8   day_of_week              368424 non-null  int32         
 9   month                    368424 non-null  int32         
 10  is_weekend               368424 non-null  int64         
 11  conditions               368424 non-null  str           
 12  temperature_c            36

In [96]:
print("Original total records:", len(registered) + len(direct))
print("Aggregated total count:", final_data["total_count"].sum())

Original total records: 3292679
Aggregated total count: 3292679


In [97]:
# confirm all holidays matched
matched_holiday_dates = final_data[final_data["is_holiday"] == 1]["date"].nunique()
original_holiday_dates = holidays["date"].nunique()

print("Matched holiday dates:", matched_holiday_dates)
print("Original holiday dates:", original_holiday_dates)

Matched holiday dates: 21
Original holiday dates: 21


## 6. Save final dataset

In [99]:
# a full version with holiday names
final_data.to_csv("../data/final_data_with_holiday_jp.csv", index=False)


# a model-ready version 
model_data = final_data.drop(columns=["holiday"])
model_data.to_csv("../data/final_model_data_jp.csv", index=False)

## 7. Conclusion
In this notebook, the raw bike rental event records is transformed into a complete location-level hourly demand dataset. Each row in the final dataset represents the rental demand for one location during one specific hourly time window.

The registered rental records and direct pickup records were combined into one event table. Then the data was grouped by `hour`, `location_id`, and `rental_type`. To make the dataset complete, I created a full hour-location table using all hourly timestamps and all observed rental locations. This allows me to explicitly add missing location-hour records. These missing rental rows were later marked and filled with `0`. Additional time-based features including `date`, `hour_of_day`, `day_of_week`, `month`, and `is_weekend` were added. 

Then I enriched the dataset with hourly weather information. Missing weather values were treated differently: numeric weather features were filled using interpolation with forward and backward filling, while categorical weather features were filled using forward and backward filling only. 

Finally, holiday information was merged by date.

The final output consists of two CSV files: a full version with the holiday name column for checking and validation, and a model-ready version without the holiday text column. This final dataset is ready to be migrated into the Dagster asset pipeline.